# LeetCode 529 — Minesweeper bằng Python

Link đề: https://leetcode.com/problems/minesweeper/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def updateBoard(self, board: List[List[str]], click: List[int]) -> List[List[str]]:
        ...
```

Ghi chú: bài **có một đáp án duy nhất** cho mỗi input (luật lật ô hoàn toàn xác định, không có ngẫu nhiên), nên ta có thể so sánh bảng kết quả bằng `==`. Ngoài ra notebook còn dựng thêm một **hàm kiểm tra tính hợp lệ** độc lập để bắt các lỗi mà việc so với một đáp án cho trước không phát hiện hết, và so sánh nhiều cách giải trên rất nhiều bảng ngẫu nhiên.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu luật lật ô của trò **Dò mìn (Minesweeper)** và cách đề biểu diễn bảng bằng ma trận ký tự `M, E, B, 1..8, X`.
2. Nắm các công cụ Python: ma trận list-of-list, duyệt **8 hướng lân cận** bằng danh sách `DIRS`, kiểm tra biên, `collections.deque`, đệ quy và **giới hạn độ sâu đệ quy**, `copy.deepcopy` (tránh sửa nhầm bảng gốc).
3. Đi từ **quét lặp tới khi không còn đổi (brute force)** → **DFS đệ quy** → **BFS bằng hàng đợi** → **tiền xử lý đếm mìn + Union-Find (thành phần liên thông các ô trống)**.
4. Hiểu mấu chốt: đây là bài **loang (flood fill)** mà chỉ ô `'B'` (không có mìn lân cận) mới tiếp tục lan; ô có số là **biên** dừng lại. Việc ghi giá trị mới vào ô cũng chính là đánh dấu "đã thăm".
5. Biết test bài toán bảng: so sánh bảng kết quả giữa nhiều cách, viết hàm kiểm tra hợp lệ độc lập, và nhận ra các lỗi kinh điển (chỉ 4 hướng, quên đánh dấu, lan tiếp qua ô có số, ghi số nguyên thay vì chuỗi, đệ quy quá sâu).

## 2. Đề bài & ví dụ

Cho ma trận ký tự `board` kích thước `m x n` biểu diễn bảng trò Dò mìn:

- `'M'`: một **mìn chưa lật**.
- `'E'`: một **ô trống chưa lật**.
- `'B'`: một ô trống **đã lật** và **không có mìn nào** ở 8 ô xung quanh (trên, dưới, trái, phải và 4 đường chéo).
- chữ số `'1'` đến `'8'`: ô đã lật, cho biết có bao nhiêu mìn ở 8 ô xung quanh.
- `'X'`: một **mìn đã lật**.

Cho thêm `click = [r, c]` là ô người chơi bấm tiếp theo, đảm bảo ô đó **chưa lật** (`'M'` hoặc `'E'`).

Hãy trả về bảng sau khi lật ô đó theo luật:

1. Nếu lật trúng mìn `'M'`: trò chơi kết thúc, đổi nó thành `'X'`.
2. Nếu lật ô `'E'` **không** có mìn xung quanh: đổi thành `'B'` và **lật đệ quy** mọi ô chưa lật xung quanh nó.
3. Nếu lật ô `'E'` có **ít nhất một** mìn xung quanh: đổi thành chữ số (`'1'`..`'8'`) bằng số mìn xung quanh.
4. Trả về bảng khi không còn ô nào cần lật nữa.

**Ví dụ 1:**

```
Input:  board = [["E","E","E","E","E"],
                 ["E","E","M","E","E"],
                 ["E","E","E","E","E"],
                 ["E","E","E","E","E"]],  click = [3,0]
Output: [["B","1","E","1","B"],
         ["B","1","M","1","B"],
         ["B","1","1","1","B"],
         ["B","B","B","B","B"]]
```

**Ví dụ 2:**

```
Input:  board = [["B","1","E","1","B"],
                 ["B","1","M","1","B"],
                 ["B","1","1","1","B"],
                 ["B","B","B","B","B"]],  click = [1,2]
Output: [["B","1","E","1","B"],
         ["B","1","X","1","B"],
         ["B","1","1","1","B"],
         ["B","B","B","B","B"]]
```

**Ràng buộc:** `1 <= m, n <= 50`; mỗi ô là `'M'`, `'E'`, `'B'` hoặc chữ số `'1'..'8'`; `click` hợp lệ và `board[click_r][click_c]` là `'M'` hoặc `'E'`.

**Đáp án có duy nhất không?** Có. Luật lật hoàn toàn xác định nên mỗi cặp `(board, click)` cho đúng một bảng kết quả. Vì thế ta có thể so sánh bằng `==` (khác với các bài ngẫu nhiên). Lưu ý đề cho phép sửa **tại chỗ** rồi trả lại chính `board`.

In [1]:
import sys
import time
import random
from collections import deque, Counter
from copy import deepcopy
from typing import List


def board_from(rows):
    """Đổi list các chuỗi thành ma trận list-of-list ký tự: ["EEM"] -> [["E","E","M"]]."""
    return [list(r) for r in rows]


def show(board, title=None):
    if title:
        print(title)
    for row in board:
        print(" ".join(row))


def show_side_by_side(before, after, left="trước", right="sau"):
    w = 2 * len(before[0]) - 1
    print(f"{left:<{w}}    {right}")
    for r1, r2 in zip(before, after):
        print(" ".join(r1) + "    " + " ".join(r2))


board1 = board_from(["EEEEE", "EEMEE", "EEEEE", "EEEEE"])
expected1 = board_from(["B1E1B", "B1M1B", "B111B", "BBBBB"])
board2 = deepcopy(expected1)
expected2 = board_from(["B1E1B", "B1X1B", "B111B", "BBBBB"])

show(board1, "Ví dụ 1 - bảng ban đầu, bấm vào (3, 0):")
print()
show_side_by_side(board1, expected1, "ban đầu", "kết quả mong đợi")
print()
show_side_by_side(board2, expected2, "ban đầu", "kết quả mong đợi (bấm (1,2) trúng mìn)")

Ví dụ 1 - bảng ban đầu, bấm vào (3, 0):
E E E E E
E E M E E
E E E E E
E E E E E

ban đầu      kết quả mong đợi
E E E E E    B 1 E 1 B
E E M E E    B 1 M 1 B
E E E E E    B 1 1 1 B
E E E E E    B B B B B

ban đầu      kết quả mong đợi (bấm (1,2) trúng mìn)
B 1 E 1 B    B 1 E 1 B
B 1 M 1 B    B 1 X 1 B
B 1 1 1 B    B 1 1 1 B
B B B B B    B B B B B


## 3. Kiến thức nền

### 3.1 Ma trận ký tự (list of list) và việc sửa tại chỗ

Bảng là một **list chứa các list**: `board[i][j]` là ô ở hàng `i`, cột `j` (đếm từ 0). `len(board)` là số hàng `m`, `len(board[0])` là số cột `n`. Mỗi ô là một chuỗi có đúng một ký tự, nên chữ số phải là chuỗi `'3'` (không phải số `3`). Muốn đổi số thành chuỗi dùng `str(3)`.

Hai bảng "trông giống nhau" có thể là cùng một đối tượng (**alias**) hoặc hai bản sao. Sửa ô qua một tên thì tên kia cũng thấy đổi. Muốn bản sao độc lập phải `copy.deepcopy`.

In [2]:
b = [["E", "M"], ["E", "E"]]
print("m =", len(b), " n =", len(b[0]), " b[0][1] =", b[0][1])

alias = b                 # CÙNG một đối tượng
alias[0][0] = "X"
print("sau khi sửa alias, b =", b)

copy1 = b[:]              # sao chép NÔNG: các hàng bên trong vẫn dùng chung!
copy1[1][1] = "Z"
print("copy nông làm b đổi:", b)

copy2 = deepcopy(b)       # sao chép SÂU: độc lập hoàn toàn
copy2[0][1] = "Q"
print("deepcopy không ảnh hưởng b:", b, "| bản sao:", copy2)

print(str(3), type(str(3)), "| '3' == 3 ?", "3" == 3)

m = 2  n = 2  b[0][1] = M
sau khi sửa alias, b = [['X', 'M'], ['E', 'E']]
copy nông làm b đổi: [['X', 'M'], ['E', 'Z']]
deepcopy không ảnh hưởng b: [['X', 'M'], ['E', 'Z']] | bản sao: [['X', 'Q'], ['E', 'Z']]
3 <class 'str'> | '3' == 3 ? False


### 3.2 Tám hướng lân cận và kiểm tra biên

Một ô `(i, j)` có tối đa 8 ô xung quanh: các `(i + dx, j + dy)` với `dx, dy` thuộc `{-1, 0, 1}`, trừ cặp `(0, 0)` (chính nó). Ô ở góc hoặc cạnh có ít hơn 8 vì một số ô nằm **ngoài bảng**; phải kiểm tra `0 <= x < m` và `0 <= y < n` trước khi truy cập (truy cập `board[-1]` trong Python không báo lỗi mà lấy phần tử cuối, đây là cái bẫy kinh điển!).

In [3]:
DIRS = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if (dx, dy) != (0, 0)]
print(len(DIRS), DIRS)


def neighbors(i, j, m, n):
    """Các ô lân cận nằm TRONG bảng."""
    return [(i + dx, j + dy) for dx, dy in DIRS if 0 <= i + dx < m and 0 <= j + dy < n]


print("ô giữa (1,1) trong bảng 3x3:", len(neighbors(1, 1, 3, 3)), "láng giềng")
print("ô góc  (0,0) trong bảng 3x3:", neighbors(0, 0, 3, 3))
print("ô cạnh (0,1) trong bảng 3x3:", len(neighbors(0, 1, 3, 3)), "láng giềng")
print("cái bẫy chỉ số âm: [10,20,30][-1] =", [10, 20, 30][-1], "(không lỗi nhưng sai ý!)")

8 [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
ô giữa (1,1) trong bảng 3x3: 8 láng giềng
ô góc  (0,0) trong bảng 3x3: [(0, 1), (1, 0), (1, 1)]
ô cạnh (0,1) trong bảng 3x3: 5 láng giềng
cái bẫy chỉ số âm: [10,20,30][-1] = 30 (không lỗi nhưng sai ý!)


### 3.3 Đếm mìn lân cận

Với ô `(i, j)`, số mìn xung quanh là số láng giềng có ký tự `'M'`. Ô đã bấm trúng thì kết thúc ngay nên khi đếm ta chỉ cần quan tâm `'M'` (mìn chưa lật), vì trong input hợp lệ không có `'X'` (trò chơi chưa kết thúc).

In [4]:
def count_mines(board, i, j):
    m, n = len(board), len(board[0])
    return sum(1 for x, y in neighbors(i, j, m, n) if board[x][y] == "M")


demo = board_from(["EEEEE", "EEMEE", "EEEEE", "EEEEE"])
grid = [[count_mines(demo, i, j) for j in range(5)] for i in range(4)]
for row in grid:
    print(row)
print("Ô (1,2) là mìn; các ô quanh nó có số 1. Số này được ghi lên bảng khi ô được lật.")

[0, 1, 1, 1, 0]
[0, 1, 0, 1, 0]
[0, 1, 1, 1, 0]
[0, 0, 0, 0, 0]
Ô (1,2) là mìn; các ô quanh nó có số 1. Số này được ghi lên bảng khi ô được lật.


### 3.4 Hàng đợi `deque` (BFS) và ngăn xếp (DFS)

- **Hàng đợi** (vào trước ra trước): `deque.append(x)` thêm cuối, `popleft()` lấy đầu. Duyệt theo "lớp": gần trước, xa sau → **BFS** (Breadth-First Search, duyệt theo chiều rộng).
- **Ngăn xếp** (vào sau ra trước): một `list` với `append` và `pop()` → **DFS** (Depth-First Search, duyệt theo chiều sâu) viết không đệ quy.
- **Đệ quy**: hàm tự gọi chính nó; mỗi lần gọi chiếm một khung trên *call stack* (ngăn xếp lời gọi) của Python.

Khi duyệt đồ thị/ô lưới phải có cách tránh thăm một ô nhiều lần (đánh dấu **visited**). Bài này khéo ở chỗ: **ghi giá trị mới vào ô cũng là đánh dấu đã thăm** (ô không còn là `'E'`), nên không cần mảng `visited` riêng.

In [5]:
q = deque([1, 2, 3])
q.append(4)
print("queue :", q.popleft(), q.popleft(), list(q))      # 1 2 rồi còn [3, 4]

st = [1, 2, 3]
st.append(4)
print("stack :", st.pop(), st.pop(), st)                  # 4 3 rồi còn [1, 2]

queue : 1 2 [3, 4]
stack : 4 3 [1, 2]


### 3.5 Giới hạn độ sâu đệ quy

Python mặc định chỉ cho đệ quy sâu khoảng **1000** tầng (`sys.getrecursionlimit()`). Vượt quá sẽ gây `RecursionError`. Bảng của bài này có tới `50 × 50 = 2500` ô; nếu DFS đệ quy đi "một mạch" qua nhiều ô, độ sâu có thể vượt 1000. Ta sẽ đo thật ở mục 7 và 12. Đây là lý do BFS (hoặc DFS dùng ngăn xếp tự quản) an toàn hơn.

In [6]:
print("giới hạn mặc định:", sys.getrecursionlimit())


def dem(k):
    return 0 if k == 0 else 1 + dem(k - 1)


print(dem(500))
try:
    dem(10 ** 5)
except RecursionError:
    print("dem(100000) -> RecursionError (đệ quy quá sâu)")

giới hạn mặc định: 1000
500
dem(100000) -> RecursionError (đệ quy quá sâu)


## 4. Ý tưởng nền tảng / trực giác

Nghĩ đến trò chơi thật: khi bạn bấm vào một vùng trống mênh mông, các ô trống **lan ra như vết dầu loang** cho đến khi chạm các ô *có số* (viền quanh mìn). Ô có số hiện số nhưng **không lan tiếp**.

Tính tay ví dụ 1 (bảng 4 hàng, 5 cột, mìn ở `(1, 2)`), bấm `(3, 0)`:

1. Đếm mìn quanh `(3, 0)`: các ô `(2,0), (2,1), (3,1)` không phải mìn → `0` → đổi thành `'B'` và **lật mọi ô lân cận** `(2,0), (2,1), (3,1)`.
2. `(2, 0)`: láng giềng `(1,0), (1,1), (2,1), (3,0), (3,1)` đều không có mìn → `'B'` và lật tiếp.
3. `(2, 1)`: láng giềng gồm `(1,2)` là mìn → số `1`. Dừng, **không lan tiếp** từ ô này.
4. Tiếp tục như vậy, vùng loang dừng ở các ô có số `1` bao quanh mìn.

Kết quả: mọi ô quanh mìn thành `'1'`, các ô xa mìn thành `'B'`, còn bản thân mìn `(1,2)` và ô `(0,2)` (không nằm cạnh ô `'B'` nào) vẫn nguyên.

**Nhận xét quan trọng:** ô `(0, 2)` là `'E'` và *không* được lật! Vì nó không kề với ô `'B'` nào: mọi láng giềng của nó, `(0,1)`, `(0,3)`, `(1,1)`, `(1,2)`, `(1,3)`, đều là số `1` hoặc mìn (không lan). Chỉ ô `'B'` mới "đẩy" việc lật sang láng giềng.

In [7]:
# Nhìn bằng mắt: bảng đáp án và các ô còn là 'E'
for i, row in enumerate(expected1):
    print(" ".join(row))
left_E = [(i, j) for i in range(4) for j in range(5) if expected1[i][j] == "E"]
print("Ô chưa lật còn lại:", left_E)
print("Láng giềng của (0,2):", [(x, y, expected1[x][y]) for x, y in neighbors(0, 2, 4, 5)])

B 1 E 1 B
B 1 M 1 B
B 1 1 1 B
B B B B B
Ô chưa lật còn lại: [(0, 2)]
Láng giềng của (0,2): [(0, 1, '1'), (0, 3, '1'), (1, 1, '1'), (1, 2, 'M'), (1, 3, '1')]


## 5. Cách 1 — Brute Force: quét lặp tới khi không còn đổi

**Ý tưởng:** làm đúng luật theo nghĩa đen, không cần khéo léo. Gọi "ô lật" là ô vừa được đổi khỏi `'E'`.

1. Nếu bấm trúng mìn → `'X'`, xong.
2. Ngược lại lật ô được bấm (`reveal`: ghi chữ số, hoặc `'B'` nếu đếm được `0`).
3. **Lặp lại**: quét *toàn bộ* các ô đã lật *trong lượt này* mà đang là `'B'`; mỗi ô `'E'` kề với chúng thì lật. Nếu quét xong không lật thêm được ô nào thì dừng.

Vì mỗi vòng quét duyệt cả tập ô đã lật và mỗi vòng chỉ lan thêm một "lớp", số vòng quét có thể lên tới độ dài đường lan dài nhất.

In [8]:
def reveal_cell(board, i, j):
    """Lật một ô 'E': ghi số mìn lân cận hoặc 'B' nếu là 0. Trả về ký tự vừa ghi."""
    c = count_mines(board, i, j)
    board[i][j] = str(c) if c else "B"
    return board[i][j]


def update_board_brute(board, click):
    m, n = len(board), len(board[0])
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
        return board
    revealed = [(i, j)]                       # các ô đã lật trong lượt này
    reveal_cell(board, i, j)
    changed = True
    rounds = 0
    while changed:                            # quét lặp tới khi hết thay đổi
        changed = False
        rounds += 1
        for (a, b) in list(revealed):         # quét toàn bộ ô đã lật
            if board[a][b] != "B":
                continue
            for x, y in neighbors(a, b, m, n):
                if board[x][y] == "E":
                    reveal_cell(board, x, y)
                    revealed.append((x, y))
                    changed = True
    update_board_brute.last_rounds = rounds
    return board


b = deepcopy(board1)
res = update_board_brute(b, [3, 0])
show(res, "Kết quả cách brute force ví dụ 1:")
print("số vòng quét:", update_board_brute.last_rounds)
assert res == expected1

b = deepcopy(board2)
assert update_board_brute(b, [1, 2]) == expected2
print("ví dụ 2 OK")

Kết quả cách brute force ví dụ 1:
B 1 E 1 B
B 1 M 1 B
B 1 1 1 B
B B B B B
số vòng quét: 7
ví dụ 2 OK


**Độ phức tạp:** mỗi vòng quét tốn `O(k · 8)` (với `k` là số ô đã lật), tối đa `O(m·n·8)`; số vòng tối đa `O(m·n)` khi đường lan ngoằn ngoèo. Tổng xấu nhất khoảng `O((m·n)²)` (thực tế nhỏ hơn nhiều). Bộ nhớ `O(m·n)`.

**Nút thắt:** ta *quét lại* những ô `'B'` đã xử lý xong ở các vòng trước. Lẽ ra mỗi ô `'B'` chỉ cần "đẩy" láng giềng **một lần** rồi thôi. Làm sao biết ô nào mới cần xử lý? Dùng **danh sách chờ** (hàng đợi/ngăn xếp) — đó chính là BFS/DFS.

## 6. Ý tưởng thuật toán / bước đột phá

Ba quan sát tạo nên lời giải gọn:

1. **Ô số là biên, ô `'B'` mới lan.** Sau khi lật một ô, chỉ khi nó là `'B'` ta mới xử lý láng giềng; nếu là chữ số thì dừng.
2. **Mỗi ô chỉ cần lật một lần.** Ngay khi quyết định lật ô `(x, y)` ta đổi nó khỏi `'E'`; từ đó không ô nào nhận nó thêm nữa. Giá trị mới = đánh dấu đã thăm.
3. **Chỉ nhìn ô `'E'`.** Láng giềng là `'M'` thì không lật (nhưng nếu quanh ô có `'M'` thì ô đó đã không phải `'B'`, nên ta chẳng bao giờ lan tới `'M'`). Láng giềng đã lật (`'B'`, chữ số) bỏ qua.

Do đó thuật toán là **flood fill 8 hướng** bắt đầu từ ô được bấm, dừng ở các ô có số. Có ba cách tổ chức "danh sách chờ": đệ quy (DFS), hàng đợi (BFS), hoặc tính trước thành phần liên thông (Union-Find). Cả ba thăm mỗi ô tối đa một lần: `O(m·n)`.

## 7. Các cách tiếp theo

Cách 1 (mục 5) là brute force. Dưới đây là **Cách 2: DFS đệ quy**, **Cách 3: BFS bằng hàng đợi** (kèm biến thể DFS dùng ngăn xếp), **Cách 4: Tiền xử lý + Union-Find**.

### Cách 2 — DFS đệ quy

**Giải thích từng đoạn:**

1. Nếu `board[i][j] == 'M'`: đổi thành `'X'`, trả về.
2. Ngược lại gọi `dfs(i, j)`.
3. Trong `dfs(i, j)`: đếm mìn lân cận. Nếu `cnt > 0` ghi `str(cnt)` rồi **dừng**. Nếu `cnt == 0` ghi `'B'`, rồi với mỗi láng giềng còn là `'E'` gọi `dfs` (ô nào được ghi giá trị thì không còn là `'E'` nên không bị gọi lần hai).

In [9]:
def update_board_dfs(board, click):
    m, n = len(board), len(board[0])

    def dfs(i, j, depth):
        update_board_dfs.max_depth = max(update_board_dfs.max_depth, depth)
        cnt = count_mines(board, i, j)
        if cnt:
            board[i][j] = str(cnt)             # ô số: biên, dừng lại
            return
        board[i][j] = "B"                      # ghi ngay để đánh dấu đã thăm
        for x, y in neighbors(i, j, m, n):
            if board[x][y] == "E":
                dfs(x, y, depth + 1)

    update_board_dfs.max_depth = 0
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
    else:
        dfs(i, j, 1)
    return board


b = deepcopy(board1)
assert update_board_dfs(b, [3, 0]) == expected1
print("độ sâu đệ quy lớn nhất ở ví dụ 1:", update_board_dfs.max_depth)
b = deepcopy(board2)
assert update_board_dfs(b, [1, 2]) == expected2
print("ví dụ 2 OK")

# Bảng trống 50x50: độ sâu đệ quy thế nào? (nâng giới hạn để đo, rồi trả lại)
old = sys.getrecursionlimit()
sys.setrecursionlimit(20000)
empty = [["E"] * 50 for _ in range(50)]
update_board_dfs(empty, [0, 0])
print("bảng trống 50x50, bấm (0,0): độ sâu lớn nhất =", update_board_dfs.max_depth)
sys.setrecursionlimit(old)

độ sâu đệ quy lớn nhất ở ví dụ 1: 8
ví dụ 2 OK
bảng trống 50x50, bấm (0,0): độ sâu lớn nhất = 2499


Độ sâu có thể xấp xỉ `m·n = 2500`, vượt xa giới hạn mặc định `1000`. Trên LeetCode lời giải đệ quy này vẫn được chấp nhận vì môi trường chấm đã nâng giới hạn; nhưng chạy trên máy bạn mà không nâng giới hạn thì sẽ `RecursionError` (ta sẽ chứng kiến ở mục 12). Bản "nộp" dưới đây tự gọi `sys.setrecursionlimit`.

In [10]:
# Code nộp LeetCode cho Cách 2 (DFS đệ quy)
class SolutionDFS:
    def updateBoard(self, board: List[List[str]], click: List[int]) -> List[List[str]]:
        sys.setrecursionlimit(10000)           # bảng 50x50 có thể đệ quy tới ~2500 tầng
        m, n = len(board), len(board[0])

        def dfs(i: int, j: int) -> None:
            cnt = 0
            for x in range(i - 1, i + 2):
                for y in range(j - 1, j + 2):
                    if 0 <= x < m and 0 <= y < n and board[x][y] == "M":
                        cnt += 1
            if cnt:
                board[i][j] = str(cnt)
                return
            board[i][j] = "B"
            for x in range(i - 1, i + 2):
                for y in range(j - 1, j + 2):
                    if 0 <= x < m and 0 <= y < n and board[x][y] == "E":
                        dfs(x, y)

        i, j = click
        if board[i][j] == "M":
            board[i][j] = "X"
        else:
            dfs(i, j)
        return board

**Độ phức tạp:** mỗi ô bị lật tối đa một lần, mỗi lần đếm 8 láng giềng → thời gian `O(m·n)` (hằng số 8·2). Bộ nhớ `O(m·n)` cho call stack trong trường hợp xấu nhất.

### Cách 3 — BFS bằng hàng đợi (an toàn, không đệ quy)

**Giải thích từng đoạn:**

1. Trúng mìn → `'X'`, trả về.
2. Lật ô được bấm; nếu là chữ số thì xong. Nếu là `'B'` thì đưa vào hàng đợi.
3. Lấy một ô `'B'` ra khỏi hàng đợi; với mỗi láng giềng còn là `'E'`: lật nó *ngay lúc đưa vào hàng đợi* (đánh dấu thăm sớm) và nếu thành `'B'` thì đưa vào hàng đợi.

Điểm cần chú ý: **lật ngay khi thêm vào hàng đợi**, không đợi lúc lấy ra, nếu không một ô có thể bị thêm vào nhiều lần bởi nhiều láng giềng (xem lỗi 2 ở mục 13).

In [11]:
def update_board_bfs(board, click):
    m, n = len(board), len(board[0])
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
        return board
    if reveal_cell(board, i, j) != "B":        # ô có số: chỉ lật mình nó
        return board
    queue = deque([(i, j)])
    while queue:
        a, b = queue.popleft()                 # ô 'B' cần lan
        for x, y in neighbors(a, b, m, n):
            if board[x][y] == "E":
                if reveal_cell(board, x, y) == "B":
                    queue.append((x, y))
    return board


b = deepcopy(board1)
assert update_board_bfs(b, [3, 0]) == expected1
b = deepcopy(board2)
assert update_board_bfs(b, [1, 2]) == expected2
print("BFS khớp ví dụ 1 và 2")

# Biến thể DFS dùng ngăn xếp: chỉ đổi popleft() thành pop() và deque thành list
def update_board_stack(board, click):
    m, n = len(board), len(board[0])
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
        return board
    if reveal_cell(board, i, j) != "B":
        return board
    stack = [(i, j)]
    while stack:
        a, b = stack.pop()                     # LIFO thay cho FIFO
        for x, y in neighbors(a, b, m, n):
            if board[x][y] == "E":
                if reveal_cell(board, x, y) == "B":
                    stack.append((x, y))
    return board


b = deepcopy(board1)
assert update_board_stack(b, [3, 0]) == expected1
print("DFS bằng ngăn xếp cũng khớp")

BFS khớp ví dụ 1 và 2
DFS bằng ngăn xếp cũng khớp


In [12]:
# Code nộp LeetCode cho Cách 3 (BFS) - đây cũng là cách ta chọn làm câu trả lời chính
from collections import deque


class SolutionBFS:
    def updateBoard(self, board: List[List[str]], click: List[int]) -> List[List[str]]:
        m, n = len(board), len(board[0])
        i, j = click
        if board[i][j] == "M":
            board[i][j] = "X"
            return board

        def reveal(x: int, y: int) -> str:
            cnt = 0
            for a in range(x - 1, x + 2):
                for b in range(y - 1, y + 2):
                    if 0 <= a < m and 0 <= b < n and board[a][b] == "M":
                        cnt += 1
            board[x][y] = str(cnt) if cnt else "B"
            return board[x][y]

        if reveal(i, j) != "B":
            return board
        queue = deque([(i, j)])
        while queue:
            x, y = queue.popleft()
            for a in range(x - 1, x + 2):
                for b in range(y - 1, y + 2):
                    if 0 <= a < m and 0 <= b < n and board[a][b] == "E":
                        if reveal(a, b) == "B":
                            queue.append((a, b))
        return board

**Độ phức tạp:** thời gian `O(m·n)` (mỗi ô vào hàng đợi tối đa một lần, mỗi lần xét 8 láng giềng và đếm 8 láng giềng nữa: hằng số khoảng 16); bộ nhớ hàng đợi `O(m·n)` nhưng nằm trên heap nên không lo `RecursionError`.

### Cách 4 — Tiền xử lý số mìn + Union-Find (thành phần liên thông)

**Ý tưởng khác hẳn:** đừng "lan" trên bảng thật. Làm hai bước độc lập:

1. **Tiền xử lý:** tính `cnt[i][j]` (số mìn lân cận) cho mọi ô không phải mìn.
2. Gộp các ô có `cnt == 0` kề nhau (8 hướng) vào cùng một **tập** bằng **Union-Find (DSU, Disjoint Set Union)**: mỗi tập là một "vùng trống liên thông".

Khi bấm ô `E`:
- Nếu `cnt > 0`: chỉ ô đó thành `str(cnt)`.
- Nếu `cnt == 0`: mọi ô thuộc **cùng vùng** với nó thành `'B'`, và mọi láng giềng (không phải mìn) của các ô trong vùng đó thành chữ số `cnt` (nếu `> 0`).

Union-Find có hai thao tác: `find(x)` (tìm đại diện của tập chứa `x`, có *nén đường đi*) và `union(a, b)` (nối hai tập).

In [13]:
class DSU:
    def __init__(self, size):
        self.parent = list(range(size))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # nén đường đi (path halving)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


d = DSU(6)
d.union(0, 1); d.union(1, 2); d.union(4, 5)
print("0~2:", d.find(0) == d.find(2), "| 0~4:", d.find(0) == d.find(4), "| 4~5:", d.find(4) == d.find(5))


def update_board_dsu(board, click):
    m, n = len(board), len(board[0])
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
        return board

    cnt = [[-1 if board[a][b] == "M" else count_mines(board, a, b) for b in range(n)] for a in range(m)]
    dsu = DSU(m * n)
    for a in range(m):
        for b in range(n):
            if cnt[a][b] == 0:
                for x, y in neighbors(a, b, m, n):
                    if cnt[x][y] == 0:
                        dsu.union(a * n + b, x * n + y)

    if cnt[i][j] > 0:
        board[i][j] = str(cnt[i][j])
        return board

    root = dsu.find(i * n + j)
    for a in range(m):
        for b in range(n):
            if cnt[a][b] == 0 and dsu.find(a * n + b) == root:
                board[a][b] = "B"
                for x, y in neighbors(a, b, m, n):
                    if cnt[x][y] > 0:
                        board[x][y] = str(cnt[x][y])      # biên của vùng: ô có số
    return board


b = deepcopy(board1)
assert update_board_dsu(b, [3, 0]) == expected1
b = deepcopy(board2)
assert update_board_dsu(b, [1, 2]) == expected2
print("Union-Find khớp ví dụ 1 và 2")

0~2: True | 0~4: False | 4~5: True
Union-Find khớp ví dụ 1 và 2


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionDSU:
    def updateBoard(self, board: List[List[str]], click: List[int]) -> List[List[str]]:
        m, n = len(board), len(board[0])
        i, j = click
        if board[i][j] == "M":
            board[i][j] = "X"
            return board
        dirs = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if (dx, dy) != (0, 0)]

        def around(a, b):
            return [(a + dx, b + dy) for dx, dy in dirs if 0 <= a + dx < m and 0 <= b + dy < n]

        cnt = [[0] * n for _ in range(m)]
        for a in range(m):
            for b in range(n):
                if board[a][b] == "M":
                    cnt[a][b] = -1
                else:
                    cnt[a][b] = sum(1 for x, y in around(a, b) if board[x][y] == "M")

        parent = list(range(m * n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for a in range(m):
            for b in range(n):
                if cnt[a][b] == 0:
                    for x, y in around(a, b):
                        if cnt[x][y] == 0:
                            parent[find(a * n + b)] = find(x * n + y)

        if cnt[i][j] > 0:
            board[i][j] = str(cnt[i][j])
            return board
        root = find(i * n + j)
        for a in range(m):
            for b in range(n):
                if cnt[a][b] == 0 and find(a * n + b) == root:
                    board[a][b] = "B"
                    for x, y in around(a, b):
                        if cnt[x][y] > 0:
                            board[x][y] = str(cnt[x][y])
        return board

**Độ phức tạp:** tiền xử lý `O(m·n·8)`, gộp `O(m·n·8·α)` với `α` là hàm Ackermann ngược (coi như hằng số), gán kết quả `O(m·n·8)`; tổng `O(m·n)` nhưng **luôn** xử lý cả bảng ngay cả khi vùng lật nhỏ (BFS/DFS chỉ chạm vùng liên quan). Bộ nhớ `O(m·n)`.

Cách này hữu ích khi bạn phải **trả lời nhiều lần bấm** trên cùng một bảng mìn cố định: tiền xử lý một lần, mỗi lần bấm chỉ tra vùng.

## 8. TRACE từng bước

Ta trace **BFS** trên ví dụ 1 (`click = (3, 0)`, mìn ở `(1, 2)`). Mỗi dòng là một lần lấy một ô `'B'` ra khỏi hàng đợi. Cột "lật thêm" cho biết các láng giềng `'E'` vừa được lật (và giá trị chúng nhận). Chỉ ô `'B'` được đưa vào hàng đợi; ô số được lật nhưng dừng ngay.

| bước | ô lấy ra | lật thêm (ô → giá trị) | hàng đợi sau bước |
|---|---|---|---|
| (khởi động) | click `(3,0)` → `B` | — | `[(3,0)]` |
| 1 | `(3,0)` | `(2,0)→B, (2,1)→1, (3,1)→B` | `[(2,0), (3,1)]` |
| 2 | `(2,0)` | `(1,0)→B, (1,1)→1` | `[(3,1), (1,0)]` |
| 3 | `(3,1)` | `(2,2)→1, (3,2)→B` | `[(1,0), (3,2)]` |
| 4 | `(1,0)` | `(0,0)→B, (0,1)→1` | `[(3,2), (0,0)]` |
| 5 | `(3,2)` | `(2,3)→1, (3,3)→B` | `[(0,0), (3,3)]` |

Ô số như `(2,1)`, `(1,1)`, `(2,2)` được ghi giá trị ngay lúc được "lật thêm" và **không vào hàng đợi**, nên không lan tiếp. Chạy hàm trace để xem toàn bộ các bước và ảnh chụp bảng ở vài bước đầu:

In [15]:
def trace_bfs(board, click, snapshots=(1, 2, 3)):
    """In từng bước BFS: ô lấy ra, láng giềng vừa lật, hàng đợi; chụp bảng ở vài bước."""
    board = deepcopy(board)
    m, n = len(board), len(board[0])
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
        show(board, "Bấm trúng mìn -> 'X', hết trò chơi:")
        return board
    first = reveal_cell(board, i, j)
    print(f"khởi động: click {tuple(click)} -> {first}")
    queue = deque([(i, j)])
    step = 0
    while queue:
        a, b = queue.popleft()
        step += 1
        added = []
        for x, y in neighbors(a, b, m, n):
            if board[x][y] == "E":
                v = reveal_cell(board, x, y)
                added.append(f"({x},{y})→{v}")
                if v == "B":
                    queue.append((x, y))
        print(f"bước {step:2d}: lấy {(a, b)} = B | lật thêm: {', '.join(added) if added else '-':32s} | hàng đợi: {list(queue)}")
        if step in snapshots:
            show(board, f"  bảng sau bước {step}:")
    show(board, "Bảng cuối cùng:")
    return board


final = trace_bfs(board1, [3, 0])
assert final == expected1

khởi động: click (3, 0) -> B
bước  1: lấy (3, 0) = B | lật thêm: (2,0)→B, (2,1)→1, (3,1)→B        | hàng đợi: [(2, 0), (3, 1)]
  bảng sau bước 1:
E E E E E
E E M E E
B 1 E E E
B B E E E
bước  2: lấy (2, 0) = B | lật thêm: (1,0)→B, (1,1)→1                 | hàng đợi: [(3, 1), (1, 0)]
  bảng sau bước 2:
E E E E E
B 1 M E E
B 1 E E E
B B E E E
bước  3: lấy (3, 1) = B | lật thêm: (2,2)→1, (3,2)→B                 | hàng đợi: [(1, 0), (3, 2)]
  bảng sau bước 3:
E E E E E
B 1 M E E
B 1 1 E E
B B B E E
bước  4: lấy (1, 0) = B | lật thêm: (0,0)→B, (0,1)→1                 | hàng đợi: [(3, 2), (0, 0)]
bước  5: lấy (3, 2) = B | lật thêm: (2,3)→1, (3,3)→B                 | hàng đợi: [(0, 0), (3, 3)]
bước  6: lấy (0, 0) = B | lật thêm: -                                | hàng đợi: [(3, 3)]
bước  7: lấy (3, 3) = B | lật thêm: (2,4)→B, (3,4)→B                 | hàng đợi: [(2, 4), (3, 4)]
bước  8: lấy (2, 4) = B | lật thêm: (1,3)→1, (1,4)→B                 | hàng đợi: [(3, 4), (1, 4)]
bước  9: lấy (3, 4

**Trace DFS đệ quy** (thụt lề theo độ sâu) trên bảng nhỏ hơn để dễ theo dõi: bảng 3x4, mìn ở `(2, 1)`, bấm `(0, 3)` (chính là bài tập 2 ở mục 17). Mỗi dòng: ô đang xét, số mìn đếm được và hành động.

In [16]:
def trace_dfs(board, click):
    board = deepcopy(board)
    m, n = len(board), len(board[0])

    def dfs(i, j, depth):
        pad = "  " * depth
        cnt = count_mines(board, i, j)
        if cnt:
            board[i][j] = str(cnt)
            print(f"{pad}({i},{j}): đếm được {cnt} mìn -> ghi '{cnt}', DỪNG (ô số là biên)")
            return
        board[i][j] = "B"
        print(f"{pad}({i},{j}): 0 mìn -> ghi 'B', lan sang các ô 'E' kề")
        for x, y in neighbors(i, j, m, n):
            if board[x][y] == "E":
                dfs(x, y, depth + 1)

    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
    else:
        dfs(i, j, 0)
    return board


board3 = board_from(["EEEE", "EEEE", "EMEE"])
res3 = trace_dfs(board3, [0, 3])
show(res3, "Kết quả:")

(0,3): 0 mìn -> ghi 'B', lan sang các ô 'E' kề
  (0,2): 0 mìn -> ghi 'B', lan sang các ô 'E' kề
    (0,1): 0 mìn -> ghi 'B', lan sang các ô 'E' kề
      (0,0): 0 mìn -> ghi 'B', lan sang các ô 'E' kề
        (1,0): đếm được 1 mìn -> ghi '1', DỪNG (ô số là biên)
        (1,1): đếm được 1 mìn -> ghi '1', DỪNG (ô số là biên)
      (1,2): đếm được 1 mìn -> ghi '1', DỪNG (ô số là biên)
    (1,3): 0 mìn -> ghi 'B', lan sang các ô 'E' kề
      (2,2): đếm được 1 mìn -> ghi '1', DỪNG (ô số là biên)
      (2,3): 0 mìn -> ghi 'B', lan sang các ô 'E' kề
Kết quả:
B B B B
1 1 1 B
E M 1 B


**Bảng trạng thái nhìn tổng quát** (cho mọi thuật toán): với mỗi ô, trạng thái đi theo đường `E → (B | chữ số)` và **không bao giờ quay lại**. Điều đó bảo đảm dừng: mỗi bước làm giảm số ô `'E'` đi 1.

| trạng thái ô | ý nghĩa | còn được xử lý? |
|---|---|---|
| `E` | chưa lật | có thể bị lật khi kề với một ô `B` |
| `B` (vừa lật) | lật xong, 0 mìn kề | **có**: lan sang các láng giềng `E` (đúng một lần) |
| `1`..`8` | lật xong | **không**: ô biên |
| `M` | mìn chưa lật | không bao giờ bị lật khi lan (vì ô kề mìn đã là số) |
| `X` | mìn bị bấm trúng | kết thúc |

## 9. Code nộp LeetCode

Chọn **BFS** làm đáp án chính: `O(m·n)`, không phụ thuộc giới hạn đệ quy. Giữ nguyên `class Solution`, `updateBoard(self, board, click)` và trả về chính `board`:

In [17]:
from collections import deque
from typing import List


class Solution:
    def updateBoard(self, board: List[List[str]], click: List[int]) -> List[List[str]]:
        m, n = len(board), len(board[0])
        i, j = click

        # 1. Bấm trúng mìn: đổi thành 'X' và kết thúc
        if board[i][j] == "M":
            board[i][j] = "X"
            return board

        # 2. Lật một ô 'E': ghi số mìn lân cận hoặc 'B'
        def reveal(x: int, y: int) -> str:
            cnt = 0
            for a in range(x - 1, x + 2):
                for b in range(y - 1, y + 2):
                    if 0 <= a < m and 0 <= b < n and board[a][b] == "M":
                        cnt += 1
            board[x][y] = str(cnt) if cnt else "B"
            return board[x][y]

        # 3. Ô số: dừng. Ô 'B': loang ra các ô 'E' kề bằng BFS
        if reveal(i, j) != "B":
            return board
        queue = deque([(i, j)])
        while queue:
            x, y = queue.popleft()
            for a in range(x - 1, x + 2):
                for b in range(y - 1, y + 2):
                    if 0 <= a < m and 0 <= b < n and board[a][b] == "E":
                        if reveal(a, b) == "B":       # ghi ngay -> đồng thời là đánh dấu đã thăm
                            queue.append((a, b))
        return board


# Chạy thử đúng kiểu LeetCode (nhớ deepcopy vì hàm sửa tại chỗ)
sol = Solution()
r1 = sol.updateBoard(deepcopy(board1), [3, 0])
r2 = sol.updateBoard(deepcopy(board2), [1, 2])
print(r1)
print(r2)
assert r1 == expected1 and r2 == expected2

[['B', '1', 'E', '1', 'B'], ['B', '1', 'M', '1', 'B'], ['B', '1', '1', '1', 'B'], ['B', 'B', 'B', 'B', 'B']]
[['B', '1', 'E', '1', 'B'], ['B', '1', 'X', '1', 'B'], ['B', '1', '1', '1', 'B'], ['B', 'B', 'B', 'B', 'B']]


## 10. Vì sao đúng

**Bất biến (invariant):** tại mọi thời điểm, (a) mọi ô đã bị đổi khỏi `'E'` mang đúng giá trị (chữ số bằng số mìn kề, hoặc `'B'` nếu 0 mìn); (b) mọi ô `'B'` **nằm trong hàng đợi hoặc đã xử lý xong** (đã lật hết các láng giềng `'E'` của nó).

**Khi dừng** hàng đợi rỗng, nghĩa là mọi ô `'B'` mới lật đã lật hết láng giềng `'E'` của mình. Vậy:

1. *Không bỏ sót:* một ô cần lật theo luật là ô kề một ô `'B'` được lật; thuật toán xử lý mọi `'B'` như thế nên nó sẽ được lật.
2. *Không lật thừa:* ta chỉ lật láng giềng của ô `'B'`, đúng luật; ô số không lan.
3. *Mỗi ô lật đúng một lần:* ta chỉ lật ô đang là `'E'` và đổi ngay, nên không ô nào bị lật hai lần, nên số vòng lặp `<= m·n`.
4. Ô mìn không bị đụng: ô kề một mìn có số mìn kề `>= 1` nên không phải `'B'`, nên không lan tới `'M'`. (Nếu click trúng `'M'` thì đổi `'X'` ngay.)

**Kiểm tra bằng một hàm hợp lệ độc lập.** Thay vì chỉ so với một đáp án viết sẵn, ta viết `explain(before, after, click)` kiểm tra *tính chất* của bảng kết quả mà **không dùng bất kỳ thuật toán loang nào để tạo ra kết quả**:

- (P1) Nếu bấm `'M'`: kết quả = bảng cũ với ô đó thành `'X'`.
- (P2) Mọi ô đổi giá trị phải từng là `'E'` và nhận **đúng** ký tự (chữ số = số `'M'` kề trong bảng cũ, hoặc `'B'`).
- (P3) Ô được bấm phải đổi.
- (P4) **Đóng kín:** trong bảng mới không còn ô `'E'` nào kề một ô `'B'`.
- (P5) **Liên thông:** mọi ô đổi phải tới được từ ô bấm qua dãy ô đổi, mà mọi ô trung gian là `'B'`.

(P4) bắt "bỏ sót", (P5) bắt "lật thừa", (P2) bắt "ghi sai giá trị".

In [18]:
def explain(before, after, click):
    """None nếu `after` đúng là kết quả hợp lệ; ngược lại trả chuỗi mô tả lỗi."""
    m, n = len(before), len(before[0])
    if len(after) != m or any(len(r) != n for r in after):
        return "kích thước bảng đổi"
    i, j = click
    if before[i][j] == "M":                                   # P1
        want = deepcopy(before)
        want[i][j] = "X"
        return None if after == want else "bấm mìn: kết quả phải là bảng cũ với ô bấm thành 'X'"
    changed = {(a, b) for a in range(m) for b in range(n) if before[a][b] != after[a][b]}
    if (i, j) not in changed:                                 # P3
        return "ô được bấm không đổi"
    for a, b in changed:                                      # P2
        if before[a][b] != "E":
            return f"ô {(a, b)} không phải 'E' mà bị đổi"
        c = sum(1 for x, y in neighbors(a, b, m, n) if before[x][y] == "M")
        want_ch = str(c) if c else "B"
        if after[a][b] != want_ch:
            return f"ô {(a, b)} phải là '{want_ch}' nhưng là {after[a][b]!r}"
    for a in range(m):                                        # P4
        for b in range(n):
            if after[a][b] == "E" and any(after[x][y] == "B" for x, y in neighbors(a, b, m, n)):
                return f"ô {(a, b)} vẫn là 'E' dù kề một ô 'B' (bỏ sót)"
    seen, stack = {(i, j)}, [(i, j)]                          # P5
    while stack:
        a, b = stack.pop()
        if after[a][b] == "B":
            for x, y in neighbors(a, b, m, n):
                if (x, y) in changed and (x, y) not in seen:
                    seen.add((x, y))
                    stack.append((x, y))
    if seen != changed:
        return f"các ô {sorted(changed - seen)} bị lật nhưng không liên thông với ô bấm (lật thừa)"
    return None


def is_valid(before, after, click):
    return explain(before, after, click) is None


assert is_valid(board1, expected1, [3, 0])
assert is_valid(board2, expected2, [1, 2])
# thử "phá" kết quả đúng để thấy hàm bắt lỗi
bad = deepcopy(expected1); bad[0][2] = "B"
print(explain(board1, bad, [3, 0]))
bad = deepcopy(expected1); bad[2][2] = "2"
print(explain(board1, bad, [3, 0]))
bad = deepcopy(expected1); bad[0][2] = "E"; bad[0][1] = "E"
print(explain(board1, bad, [3, 0]))

ô (0, 2) phải là '1' nhưng là 'B'
ô (2, 2) phải là '1' nhưng là '2'
ô (0, 1) vẫn là 'E' dù kề một ô 'B' (bỏ sót)


## 11. Time / Space Complexity

Đặt `m × n` là kích thước bảng, `N = m·n`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Quét lặp (brute) | `O(N²)` xấu nhất | `O(N)` | quét lại ô cũ nhiều lần |
| 2. DFS đệ quy | `O(N)` | `O(N)` call stack | độ sâu tới `~N` > giới hạn 1000 của Python |
| 3. BFS / DFS ngăn xếp | `O(N)` | `O(N)` hàng đợi | an toàn, không đệ quy |
| 4. Union-Find | `O(N · α(N))` ≈ `O(N)` | `O(N)` | luôn xử lý cả bảng |

Mỗi ô bị lật tối đa một lần và mỗi lần ta xét 8 láng giềng (cộng đếm mìn 8 láng giềng nữa), nên hằng số cỡ 16. Đo thực tế trên bảng `50 × 50` ít mìn (vùng lan rất lớn):

In [19]:
def random_board(rng, m, n, p):
    """Bảng m x n; mỗi ô là mìn 'M' với xác suất p, ngược lại 'E'."""
    return [["M" if rng.random() < p else "E" for _ in range(n)] for _ in range(m)]


def first_empty(board):
    for i, row in enumerate(board):
        for j, v in enumerate(row):
            if v == "E":
                return [i, j]


def timeit(fn, board, click, repeat=5):
    best = 1e9
    for _ in range(repeat):
        b = deepcopy(board)
        t0 = time.perf_counter()
        fn(b, click)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(2024)
big = random_board(rng, 50, 50, 0.01)
click = first_empty(big)
old = sys.getrecursionlimit()
sys.setrecursionlimit(20000)
print("bảng 50x50, ~1% mìn, số mìn =", sum(r.count("M") for r in big), " click =", click)
for name, fn in [("brute", update_board_brute), ("dfs", update_board_dfs), ("bfs", update_board_bfs),
                 ("stack", update_board_stack), ("dsu", update_board_dsu)]:
    print(f"  {name:6s} {timeit(fn, big, click) * 1000:8.2f} ms")
sys.setrecursionlimit(old)
print("(thời gian chỉ mang tính tham khảo)")

bảng 50x50, ~1% mìn, số mìn = 18  click = [0, 0]


  brute     69.73 ms
  dfs        8.60 ms
  bfs        7.64 ms
  stack      7.11 ms


  dsu       17.65 ms
(thời gian chỉ mang tính tham khảo)


## 12. Bộ test

Chia làm 6 nhóm: (A) ví dụ của đề; (B) edge case; (C) phản ví dụ: cài đặt sai phải bị bắt; (D) đối chiếu các cách và hàm hợp lệ trên **bảng ngẫu nhiên có seed**; (E) bảng kết quả so sánh; (F) giới hạn lớn nhất `50 × 50` và đệ quy.

### A. Ví dụ của đề

Mỗi cách giải đều được chạy trên **bản sao** của input (vì hàm sửa tại chỗ).

In [20]:
IMPLS = {
    "brute": update_board_brute,
    "dfs": update_board_dfs,
    "bfs": update_board_bfs,
    "stack": update_board_stack,
    "dsu": update_board_dsu,
    "Solution(BFS)": lambda b, c: Solution().updateBoard(b, c),
    "SolutionDFS": lambda b, c: SolutionDFS().updateBoard(b, c),
    "SolutionBFS": lambda b, c: SolutionBFS().updateBoard(b, c),
    "SolutionDSU": lambda b, c: SolutionDSU().updateBoard(b, c),
}


def run_all(board, click):
    """Chạy mọi cách trên bản sao; trả dict tên -> bảng kết quả."""
    return {name: fn(deepcopy(board), list(click)) for name, fn in IMPLS.items()}


for board, click, want in [(board1, [3, 0], expected1), (board2, [1, 2], expected2)]:
    results = run_all(board, click)
    for name, got in results.items():
        assert got == want, name
        assert is_valid(board, got, click), name
print("Ví dụ 1 và 2: cả", len(IMPLS), "cài đặt đều ra đúng bảng của đề")

# hàm sửa tại chỗ: trả về CHÍNH đối tượng board đã truyền vào
b = deepcopy(board1)
assert Solution().updateBoard(b, [3, 0]) is b
print("trả về chính đối tượng đầu vào: OK")

Ví dụ 1 và 2: cả 9 cài đặt đều ra đúng bảng của đề
trả về chính đối tượng đầu vào: OK


### B. Edge case

- Bảng `1 × 1`: `'E'` → `'B'`; `'M'` → `'X'`.
- Bảng một hàng, một cột.
- Bấm ô có số (không loang).
- Bấm ô `'E'` ở góc kề mìn.
- Bấm `'M'` giữa nhiều mìn.
- Bảng toàn `'E'` (không mìn) → toàn `'B'`.
- Bảng có ô đã lật sẵn (ô `'B'`, chữ số).
- Mìn bao kín một ô `'E'` (ô đó không bao giờ bị lan tới, nhưng nếu bấm trực tiếp thì thành chữ số 8).

In [21]:
edge_cases = [
    (["E"], [0, 0], ["B"]),
    (["M"], [0, 0], ["X"]),
    (["EEEEE"], [0, 0], ["BBBBB"]),
    (["EEMEE"], [0, 0], ["B1MEE"]),
    (["E", "E", "M", "E"], [3, 0], ["E", "E", "M", "1"]),
    (["E", "E", "M", "E", "E"], [0, 0], ["B", "1", "M", "E", "E"]),
    (["EEE", "EEE", "EEE"], [1, 1], ["BBB", "BBB", "BBB"]),
    (["MEM", "EEE", "MEM"], [1, 1], ["MEM", "E4E", "MEM"]),
    (["MMM", "MEM", "MMM"], [1, 1], ["MMM", "M8M", "MMM"]),     # bị mìn bao kín: số 8
    (["MMM", "MEM", "MMM"], [0, 1], ["MXM", "MEM", "MMM"]),
]
for rows, click, want in edge_cases:
    board = board_from(rows)
    results = run_all(board, click)
    for name, got in results.items():
        assert got == board_from(want), (name, rows, click, got)
        assert is_valid(board, got, click)
print(len(edge_cases), "edge case viết tay: cả", len(IMPLS), "cài đặt đều đúng")

# một trường hợp không viết tay đáp án: dùng brute làm oracle + hàm hợp lệ
rows, click = ["EEEE", "EMEE", "EEEE"], [2, 3]
board = board_from(rows)
results = run_all(board, click)
ref = results["brute"]
show_side_by_side(board, ref, "ban đầu", "kết quả")
assert all(r == ref and is_valid(board, r, click) for r in results.values())

# bảng có ô đã lật sẵn: lật tiếp vùng còn lại
rows = ["B1EEE", "B1MEE", "B1EEE"]
board = board_from(rows)
results = run_all(board, [0, 4])
for name, got in results.items():
    assert got == results["brute"], name
    assert is_valid(board, got, [0, 4]), name
show_side_by_side(board, results["brute"], "ban đầu", "bấm (0,4)")

10 edge case viết tay: cả 9 cài đặt đều đúng
ban đầu    kết quả
E E E E    E E 1 B
E M E E    E M 1 B
E E E E    E E 1 B
ban đầu      bấm (0,4)
B 1 E E E    B 1 E 1 B
B 1 M E E    B 1 M 1 B
B 1 E E E    B 1 E 1 B


### C. Phản ví dụ: cài đặt sai phải bị bắt

Một bộ test tốt phải **thất bại** khi code sai. Ta viết các bản sai (chi tiết ở mục 13) và chắc chắn `explain` phát hiện:

In [22]:
def bug_four_dirs(board, click):
    """Sai: chỉ dùng 4 hướng (bỏ đường chéo) cho cả đếm mìn lẫn loang."""
    m, n = len(board), len(board[0])
    d4 = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
        return board
    stack = [(i, j)]
    seen = {(i, j)}
    while stack:
        a, b = stack.pop()
        c = sum(1 for dx, dy in d4 if 0 <= a + dx < m and 0 <= b + dy < n and board[a + dx][b + dy] == "M")
        board[a][b] = str(c) if c else "B"
        if not c:
            for dx, dy in d4:
                x, y = a + dx, b + dy
                if 0 <= x < m and 0 <= y < n and board[x][y] == "E" and (x, y) not in seen:
                    seen.add((x, y))
                    stack.append((x, y))
    return board


def bug_negative_index(board, click):
    """Sai: quên kiểm tra biên dưới (x >= 0): chỉ số -1 'quấn' sang hàng/cột cuối."""
    m, n = len(board), len(board[0])
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
        return board

    def reveal(x, y):
        c = sum(1 for dx, dy in DIRS if x + dx < m and y + dy < n and board[x + dx][y + dy] == "M")
        board[x][y] = str(c) if c else "B"
        return board[x][y]

    if reveal(i, j) != "B":
        return board
    q = deque([(i, j)])
    while q:
        a, b = q.popleft()
        for dx, dy in DIRS:
            x, y = a + dx, b + dy
            if x < m and y < n and board[x][y] == "E":
                if reveal(x, y) == "B":
                    q.append((x, y))
    return board


def bug_expand_through_numbers(board, click):
    """Sai: ô số vẫn lan tiếp sang láng giềng 'E'."""
    m, n = len(board), len(board[0])
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
        return board

    def dfs(a, b):
        c = count_mines(board, a, b)
        board[a][b] = str(c) if c else "B"
        for x, y in neighbors(a, b, m, n):          # SAI: không dừng khi c > 0
            if board[x][y] == "E":
                dfs(x, y)

    dfs(i, j)
    return board


def bug_only_click_cell(board, click):
    """Sai: chỉ lật đúng ô được bấm, không loang."""
    i, j = click
    if board[i][j] == "M":
        board[i][j] = "X"
    else:
        reveal_cell(board, i, j)
    return board


def bug_wrong_digit(board, click):
    """Sai: số mìn bị tính thêm 1 khi ô nằm ở cạnh (nhầm đếm cả ô ngoài bảng)."""
    board = update_board_bfs(board, click)
    for i, row in enumerate(board):
        for j, v in enumerate(row):
            if v.isdigit() and (i == 0 or j == 0):
                row[j] = str(int(v) + 1)
    return board


BUGS = [bug_four_dirs, bug_negative_index, bug_expand_through_numbers, bug_only_click_cell, bug_wrong_digit]
caught = Counter()
for rows, click in [(["EEEEE", "EEMEE", "EEEEE", "EEEEE"], [3, 0]),
                    (["EEEE", "EEEE", "EMEE"], [0, 3]),
                    (["EEEEE", "EEEEE", "EEEEM"], [0, 0]),
                    (["EEE", "EEE", "MEE"], [0, 2])]:
    board = board_from(rows)
    for bug in BUGS:
        b = deepcopy(board)
        old = sys.getrecursionlimit()
        sys.setrecursionlimit(5000)
        try:
            out = bug(b, click)
            reason = explain(board, out, click)
        except Exception as e:                      # lỗi chạy cũng tính là bị bắt
            reason = f"ngoại lệ {type(e).__name__}"
        finally:
            sys.setrecursionlimit(old)
        if reason:
            caught[bug.__name__] += 1
for bug in BUGS:
    print(f"{bug.__name__:28s} bị bắt trên {caught[bug.__name__]}/4 bảng thử")
    assert caught[bug.__name__] >= 1, bug.__name__

bug_four_dirs                bị bắt trên 4/4 bảng thử
bug_negative_index           bị bắt trên 3/4 bảng thử
bug_expand_through_numbers   bị bắt trên 2/4 bảng thử
bug_only_click_cell          bị bắt trên 4/4 bảng thử
bug_wrong_digit              bị bắt trên 3/4 bảng thử


### D. Đối chiếu các cách trên bảng ngẫu nhiên có seed

Ta sinh bảng ngẫu nhiên **đúng ràng buộc** (kích thước `1..8`, mật độ mìn từ thưa đến dày), rồi:

1. **Chơi vài nước đi hợp lệ trước** (bấm ngẫu nhiên các ô `'E'` bằng BFS) để tạo ra bảng đã lật một phần (có `'B'` và chữ số). Như vậy input giống thật (mọi `'B'` đã bị lật hết láng giềng).
2. Chọn ngẫu nhiên một ô chưa lật (`'M'` hoặc `'E'`) làm `click`.
3. Chạy cả 9 cài đặt, `assert` mọi kết quả **bằng nhau** và `is_valid`.

In [23]:
def random_partial_board(rng, m, n, p, plays):
    board = random_board(rng, m, n, p)
    for _ in range(plays):
        empties = [(i, j) for i in range(m) for j in range(n) if board[i][j] == "E"]
        if not empties:
            break
        i, j = rng.choice(empties)
        update_board_bfs(board, [i, j])
    return board


rng = random.Random(20240529)
cases = 0
shapes = Counter()
for trial in range(600):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    p = rng.choice([0.0, 0.05, 0.1, 0.2, 0.35, 0.6])
    board = random_partial_board(rng, m, n, p, rng.randint(0, 3))
    unrevealed = [(i, j) for i in range(m) for j in range(n) if board[i][j] in "ME"]
    if not unrevealed:
        continue
    click = list(rng.choice(unrevealed))
    results = run_all(board, click)
    ref = results["brute"]
    for name, got in results.items():
        assert got == ref, (name, board, click)
    assert is_valid(board, ref, click), explain(board, ref, click)
    cases += 1
    shapes["bấm mìn" if board[click[0]][click[1]] == "M" else "bấm ô trống"] += 1
print(f"{cases} bảng ngẫu nhiên: 9 cài đặt cho kết quả giống hệt nhau và hợp lệ", dict(shapes))

443 bảng ngẫu nhiên: 9 cài đặt cho kết quả giống hệt nhau và hợp lệ {'bấm ô trống': 236, 'bấm mìn': 207}


### E. Bảng kết quả so sánh giữa các cách

In kết quả của từng cách trên cùng một bảng ngẫu nhiên có vùng lật lớn, đặt cạnh nhau để thấy chúng trùng khớp (và các số khác nhau chỉ nằm ở ô *chưa* bị lật):

In [24]:
def show_many(boards, labels):
    heights = len(boards[0])
    widths = [2 * len(b[0]) - 1 for b in boards]
    print("    ".join(f"{lab:<{w}}" for lab, w in zip(labels, widths)))
    for r in range(heights):
        print("    ".join(" ".join(b[r]) for b in boards))


rng = random.Random(7)
while True:                                   # tìm (có seed, nên lặp lại được) một bảng có vùng lật khá lớn
    demo = random_board(rng, 7, 9, 0.07)
    click = first_empty(demo)
    results = run_all(demo, click)
    if sum(1 for a in range(7) for b in range(9) if demo[a][b] != results["bfs"][a][b]) >= 15:
        break
print("click =", click, "| số ô bị lật mỗi cách:")
names = ["brute", "dfs", "bfs", "dsu"]
changed_counts = {nm: sum(1 for a in range(7) for b in range(9) if demo[a][b] != results[nm][a][b]) for nm in results}
print(changed_counts)
show_many([demo] + [results[nm] for nm in names], ["ban đầu"] + names)
assert len(set(map(str, results.values()))) == 1
assert len(set(changed_counts.values())) == 1
print("\nTất cả cách cho cùng một bảng và cùng số ô bị lật")

click = [0, 0] | số ô bị lật mỗi cách:
{'brute': 45, 'dfs': 45, 'bfs': 45, 'stack': 45, 'dsu': 45, 'Solution(BFS)': 45, 'SolutionDFS': 45, 'SolutionBFS': 45, 'SolutionDSU': 45}
ban đầu              brute                dfs                  bfs                  dsu              
E E E E E E E M E    B B B B B 1 E M E    B B B B B 1 E M E    B B B B B 1 E M E    B B B B B 1 E M E
E E E E E E M E E    1 1 1 B B 1 M E E    1 1 1 B B 1 M E E    1 1 1 B B 1 M E E    1 1 1 B B 1 M E E
E M E E E E E E E    E M 1 B B 1 1 E E    E M 1 B B 1 1 E E    E M 1 B B 1 1 E E    E M 1 B B 1 1 E E
E E E E E E E E E    1 1 1 B B B 1 E E    1 1 1 B B B 1 E E    1 1 1 B B B 1 E E    1 1 1 B B B 1 E E
E E E E E E E M E    B B B B B B 2 M E    B B B B B B 2 M E    B B B B B B 2 M E    B B B B B B 2 M E
E E E E E E E M E    B B B B B B 3 M E    B B B B B B 3 M E    B B B B B B 3 M E    B B B B B B 3 M E
E E E E E E E M M    B B B B B B 2 M M    B B B B B B 2 M M    B B B B B B 2 M M    B B B B B B 2 M M

Tất cả

### F. Giới hạn lớn nhất `50 × 50` và vấn đề đệ quy

Bảng toàn `'E'` là trường hợp đệ quy sâu nhất. Ta kiểm tra:

1. BFS, DFS dùng ngăn xếp và Union-Find chạy bình thường.
2. DFS đệ quy **với giới hạn mặc định 1000** quả thật báo `RecursionError`; còn bản `SolutionDFS` (tự nâng giới hạn) thì chạy tốt.

In [25]:
empty50 = [["E"] * 50 for _ in range(50)]
full_B = [["B"] * 50 for _ in range(50)]
for nm in ("bfs", "stack", "dsu", "brute", "Solution(BFS)", "SolutionDFS", "SolutionBFS", "SolutionDSU"):
    old = sys.getrecursionlimit()
    got = IMPLS[nm](deepcopy(empty50), [25, 25])
    sys.setrecursionlimit(old)
    assert got == full_B, nm
print("bảng trống 50x50: mọi cách (có nâng giới hạn đệ quy nếu cần) đều ra toàn 'B'")

# DFS đệ quy thuần với giới hạn mặc định 1000
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    update_board_dfs(deepcopy(empty50), [0, 0])
    print("không lỗi (bất ngờ)")
except RecursionError:
    print("update_board_dfs + giới hạn 1000 -> RecursionError (độ sâu cần ~2500)")
finally:
    sys.setrecursionlimit(old)

# bảng 50x50 có mìn rải rác: đối chiếu BFS / Stack / DSU / DFS(đã nâng) / brute và hàm hợp lệ
rng = random.Random(99)
for p in (0.0, 0.02, 0.08, 0.2):
    board = random_board(rng, 50, 50, p)
    click = first_empty(board)
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(20000)
    results = run_all(board, click)
    sys.setrecursionlimit(old)
    ref = results["bfs"]
    assert all(r == ref for r in results.values())
    assert is_valid(board, ref, click)
    opened = sum(1 for a in range(50) for b in range(50) if board[a][b] != ref[a][b])
    print(f"  p={p:.2f}: số mìn={sum(r.count('M') for r in board):4d}, số ô được lật={opened:4d}  -> mọi cách trùng nhau")

bảng trống 50x50: mọi cách (có nâng giới hạn đệ quy nếu cần) đều ra toàn 'B'
update_board_dfs + giới hạn 1000 -> RecursionError (độ sâu cần ~2500)


  p=0.00: số mìn=   0, số ô được lật=2500  -> mọi cách trùng nhau


  p=0.02: số mìn=  46, số ô được lật=2441  -> mọi cách trùng nhau


  p=0.08: số mìn= 197, số ô được lật=   1  -> mọi cách trùng nhau
  p=0.20: số mìn= 504, số ô được lật=   6  -> mọi cách trùng nhau


## 13. Các lỗi người mới hay gặp

Mỗi lỗi dưới đây có **code sai → vì sao sai → code đúng**. Dùng `explain` để thấy lỗi.

### Lỗi 1: chỉ xét 4 hướng (trên, dưới, trái, phải)

Quên đường chéo. Đề nói rõ "all 4 diagonals". Một mìn nằm chéo sẽ không được đếm và ô đó bị ghi `'B'` sai.

In [26]:
board = board_from(["EEE", "EEE", "EEM"])          # mìn ở góc dưới phải
want = Solution().updateBoard(deepcopy(board), [0, 0])
bad = bug_four_dirs(deepcopy(board), [0, 0])
show_side_by_side(bad, want, "CODE SAI (4 hướng)", "ĐÚNG (8 hướng)")
print("explain:", explain(board, bad, [0, 0]))

CODE SAI (4 hướng)    ĐÚNG (8 hướng)
B B B    B B B
B B 1    B 1 1
B 1 M    B 1 M
explain: ô (1, 1) phải là '1' nhưng là 'B'


**Vì sao sai:** ô `(1,1)` kề mìn `(2,2)` theo đường chéo; code 4 hướng đếm được 0 nên ghi `'B'` và lan tiếp sai chỗ. **Đúng:** dùng `DIRS` 8 hướng (hai vòng lặp `dx, dy` trong `{-1, 0, 1}` bỏ `(0,0)`), dùng chung cho đếm và lan.

### Lỗi 2: quên kiểm tra biên dưới (`x >= 0`) — chỉ số âm "quấn vòng"

Trong Python `board[-1]` là hàng cuối (không báo lỗi!). Nếu bạn chỉ kiểm tra `x < m` mà quên `x >= 0`, ô ở hàng đầu sẽ "nhìn thấy" hàng cuối như láng giềng.

In [27]:
board = board_from(["EEE", "EEE", "MEE"])       # mìn ở hàng cuối, góc trái
want = Solution().updateBoard(deepcopy(board), [0, 2])
bad = bug_negative_index(deepcopy(board), [0, 2])
show_side_by_side(bad, want, "CODE SAI (thiếu x>=0)", "ĐÚNG")
print("explain:", explain(board, bad, [0, 2]))
print("ô (0,0) có chỉ số -1 -> board[-1] =", board[-1], " (hàng cuối được coi là 'hàng trên' của hàng 0)")

CODE SAI (thiếu x>=0)    ĐÚNG
E 1 B    B B B
E 1 B    1 1 B
M 1 B    M 1 B
explain: ô (0, 1) phải là 'B' nhưng là '1'
ô (0,0) có chỉ số -1 -> board[-1] = ['M', 'E', 'E']  (hàng cuối được coi là 'hàng trên' của hàng 0)


**Vì sao sai:** `(0, 0)` có láng giềng `(-1, 0), (-1, 1)` → trở thành `(2, 0), (2, 1)` trong Python, nên mìn `(2,0)` bị đếm cho ô `(0,0)`. **Đúng:** luôn viết đầy đủ `0 <= x < m and 0 <= y < n`.

### Lỗi 3: quên đánh dấu ô đã lật → hàng đợi phình to hoặc lặp vô hạn

Nếu điều kiện thêm vào hàng đợi là "không phải mìn" thay vì "đang là `'E'`", các ô `'B'` đã lật sẽ lại bị đẩy vào hàng đợi mãi mãi (hai ô `'B'` kề nhau cứ đẩy nhau). Dưới đây có bộ đếm bảo vệ để không treo notebook.

In [28]:
def bug_no_visited(board, click, guard=20000):
    m, n = len(board), len(board[0])
    i, j = click
    queue = deque([(i, j)])
    pushes = 0
    while queue:
        a, b = queue.popleft()
        c = count_mines(board, a, b)
        board[a][b] = str(c) if c else "B"
        if c == 0:
            for x, y in neighbors(a, b, m, n):
                if board[x][y] != "M":                 # SAI: nên là == "E"
                    queue.append((x, y))
                    pushes += 1
                    if pushes > guard:
                        raise RuntimeError(f"vòng lặp không kết thúc (đã đẩy {pushes} lần, bảng chỉ có {m * n} ô)")
    return board


try:
    bug_no_visited(board_from(["EEE", "EEE", "EEE"]), [1, 1])
except RuntimeError as e:
    print("CODE SAI ->", e)


# CODE ĐÚNG: chỉ thêm ô đang là 'E' và ghi giá trị NGAY khi thêm
def update_board_good(board, click):
    return update_board_bfs(board, click)


assert update_board_good(board_from(["EEE", "EEE", "EEE"]), [1, 1]) == board_from(["BBB", "BBB", "BBB"])
print("CODE ĐÚNG -> mỗi ô được đẩy tối đa một lần")

CODE SAI -> vòng lặp không kết thúc (đã đẩy 20001 lần, bảng chỉ có 9 ô)
CODE ĐÚNG -> mỗi ô được đẩy tối đa một lần


**Vì sao sai:** không có thứ gì ngăn một ô bị xử lý lại. **Đúng:** điều kiện `board[x][y] == "E"` + ghi giá trị mới ngay khi quyết định lật. Biến thể "nhẹ" hơn của lỗi này: lật khi **lấy ra** thay vì khi **đưa vào** hàng đợi, khiến một ô vào hàng đợi nhiều lần (không vô hạn nhưng lãng phí).

### Lỗi 4: ô số vẫn lan tiếp

Ghi số xong nhưng quên `return` / quên điều kiện, nên tiếp tục lật các láng giềng của ô số. Kết quả lật quá nhiều ô, kể cả những ô lẽ ra phải còn `'E'` (như ô `(0, 2)` ở ví dụ 1).

In [29]:
bad = bug_expand_through_numbers(deepcopy(board1), [3, 0])
show_side_by_side(bad, expected1, "CODE SAI (ô số lan tiếp)", "ĐÚNG")
print("explain:", explain(board1, bad, [3, 0]))

CODE SAI (ô số lan tiếp)    ĐÚNG
B 1 1 1 B    B 1 E 1 B
B 1 M 1 B    B 1 M 1 B
B 1 1 1 B    B 1 1 1 B
B B B B B    B B B B B
explain: các ô [(0, 2)] bị lật nhưng không liên thông với ô bấm (lật thừa)


**Vì sao sai:** ô số là **biên** của vùng lật; chỉ ô `'B'` mới lan. **Đúng:** `if cnt: board[i][j] = str(cnt); return`.

### Lỗi 5: ghi số nguyên thay vì chuỗi

Bảng chứa **ký tự**. `board[i][j] = cnt` ghi số nguyên `1` (không phải `'1'`), khiến so sánh với đáp án thất bại và `" ".join(row)` văng `TypeError`.

In [30]:
b = board_from(["EM"])
cnt = count_mines(b, 0, 0)
b[0][0] = cnt                         # SAI: số nguyên
print("board =", b, "| 1 == '1' ?", 1 == "1")
try:
    print(" ".join(b[0]))
except TypeError as e:
    print("TypeError:", e)

b[0][0] = str(cnt)                    # ĐÚNG
print("sửa lại:", b, "->", " ".join(b[0]))

board = [[1, 'M']] | 1 == '1' ? False
TypeError: sequence item 0: expected str instance, int found
sửa lại: [['1', 'M']] -> 1 M


### Lỗi 6: test trên bảng đã bị sửa tại chỗ

`updateBoard` sửa **chính** `board` được truyền vào. Nếu bạn dùng lại biến `board` cho lần test sau, đầu vào lần sau đã là kết quả lần trước, nên kết quả khó hiểu (và đôi khi "đúng một cách ngẫu nhiên").

In [31]:
shared = board_from(["EEEEE", "EEMEE", "EEEEE", "EEEEE"])
r_a = Solution().updateBoard(shared, [3, 0])
print("sau lần gọi 1, biến shared đã đổi:")
show(shared)
r_b = Solution().updateBoard(shared, [0, 2])         # lần 2 chạy trên bảng đã lật rồi
print("kết quả lần 2 (ô (0,2) vẫn 'E' nên chuyển thành số):")
show(r_b)
print("r_a is shared ?", r_a is shared, "-> chỉ là MỘT đối tượng duy nhất")
print("Cách an toàn: Solution().updateBoard(deepcopy(board), click)")

sau lần gọi 1, biến shared đã đổi:
B 1 E 1 B
B 1 M 1 B
B 1 1 1 B
B B B B B
kết quả lần 2 (ô (0,2) vẫn 'E' nên chuyển thành số):
B 1 1 1 B
B 1 M 1 B
B 1 1 1 B
B B B B B
r_a is shared ? True -> chỉ là MỘT đối tượng duy nhất
Cách an toàn: Solution().updateBoard(deepcopy(board), click)


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** đề cho lưới, một ô xuất phát, luật "lan ra các ô kề thoả điều kiện" → **flood fill** (loang) bằng DFS/BFS. Tương tự: đếm đảo (200), tô màu (733), bao vây vùng (130).

**Mẹo triển khai:**

1. Khai báo `DIRS` 8 hướng một lần, viết hàm `inside(x, y)` hoặc `neighbors`.
2. Ghi giá trị mới vào ô *ngay khi* quyết định thăm; đừng đợi lúc lấy ra.
3. Phân biệt **ô lan** (`'B'`) và **ô biên** (chữ số): chỉ ô lan đưa láng giềng vào danh sách chờ.
4. Nếu lưới tới `50 × 50` và dùng đệ quy, nhớ giới hạn độ sâu; BFS tránh được.
5. Khi test bài sửa tại chỗ: luôn `deepcopy` input.
6. Test bằng **tính chất** (đóng kín, liên thông, giá trị đúng) song song với so đáp án.

**Phép đếm láng giềng 8 hướng** xuất hiện ở nhiều bài khác: Game of Life (289), Dò mìn (529, 1275...), Image Smoother (661).

## 15. Biến thể

1. **Đếm số ô được lật** sau một lần bấm (thêm biến đếm vào BFS).
2. **Nhiều lần bấm liên tiếp**: áp dụng `updateBoard` lần lượt; dừng khi gặp mìn (thua) hoặc không còn `'E'` (thắng).
3. **Bảng không chứa sẵn số**: chỉ có `'M'`/`'E'`, mọi ô lật xong ghi giá trị (đúng như bản này).
4. **Dò mìn 4 hướng** (chỉ trên/dưới/trái/phải): thay `DIRS`.
5. **Dò mìn trên lưới lục giác hay 3D**: thay tập hướng.
6. **Bấm lần đầu luôn an toàn** (như game thật): dời mìn nếu ô đầu là mìn.
7. **Cờ (flag)**: ô gắn cờ không được lật khi loang.

Ví dụ biến thể 1 và 2:

In [32]:
def reveal_with_count(board, click):
    """Biến thể 1: trả về (bảng, số ô vừa được lật)."""
    before = deepcopy(board)
    after = Solution().updateBoard(board, click)
    n_changed = sum(1 for i in range(len(after)) for j in range(len(after[0])) if before[i][j] != after[i][j])
    return after, n_changed


_, k = reveal_with_count(deepcopy(board1), [3, 0])
print("Ví dụ 1 lật", k, "ô (4x5 = 20 ô; trừ 1 mìn và ô (0,2) chưa lật)")
assert k == 18

_, k = reveal_with_count(deepcopy(board2), [1, 2])
print("Ví dụ 2 (bấm mìn) đổi", k, "ô")
assert k == 1


def play(board, clicks):
    """Biến thể 2: áp dụng lần lượt các lần bấm. Trả 'LOST' / 'WON' / 'PLAYING'."""
    board = deepcopy(board)
    for c in clicks:
        Solution().updateBoard(board, c)
        if any("X" in row for row in board):
            return "LOST"
        if not any("E" in row for row in board):
            return "WON"
    return "PLAYING"


print(play(board_from(["EEE", "EME", "EEE"]), [[0, 0]]))                 # lật ô (0,0): chỉ ra số 1
print(play(board_from(["EEE", "EME", "EEE"]), [[0, 0], [1, 1]]))         # bấm trúng mìn

Ví dụ 1 lật 18 ô (4x5 = 20 ô; trừ 1 mìn và ô (0,2) chưa lật)
Ví dụ 2 (bấm mìn) đổi 1 ô
PLAYING
LOST


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét lặp (brute) | Lặp: mọi ô `B` đã lật lan láng giềng, tới khi hết đổi | `O(N²)` xấu nhất | `O(N)` | Làm oracle để kiểm chứng; không nộp |
| 2. DFS đệ quy | Lật ô, nếu `B` thì đệ quy vào các `E` kề | `O(N)` | `O(N)` call stack (tới ~N) | Code ngắn nhất; nhớ nâng `setrecursionlimit` |
| 3. BFS (deque) / DFS ngăn xếp | Hàng đợi/ngăn xếp các ô `B`; lật ngay khi đưa vào | `O(N)` | `O(N)` heap | **Đáp án chính**: an toàn, dễ chứng minh |
| 4. Tiền xử lý + Union-Find | Đếm mìn trước, gộp ô số 0 kề nhau thành vùng | `O(N·α(N))` | `O(N)` | Nhiều lần bấm trên cùng bảng mìn; chuẩn bị cho bài cần "vùng liên thông" |

Điểm chung: cả bốn cho **cùng một bảng kết quả** (đã `assert` trên hàng trăm bảng ngẫu nhiên ở mục 12). Khác biệt nằm ở tính an toàn khi đệ quy sâu và cách tổ chức "danh sách chờ".

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO bên dưới **chưa được gọi**: bạn tự điền, rồi bỏ comment dòng test ở cuối.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [33]:
g = board_from(["MEE", "EEE", "EEM"])
print(count_mines(g, 1, 1))
print(count_mines(g, 0, 1))
print(count_mines(g, 2, 0))
print(len(neighbors(0, 0, 3, 3)), len(neighbors(1, 1, 3, 3)), len(neighbors(0, 1, 3, 3)))

2
1
0
3 8 5


### Bài 2 (tính tay)

Bảng 3 hàng, 4 cột, mìn ở `(2, 1)`:

```
E E E E
E E E E
E M E E
```

Bấm `(0, 3)`. Hãy tính tay bảng kết quả (ô nào `'B'`, ô nào số, ô nào còn `'E'`). Chú ý ô `(2, 0)`. Rồi kiểm tra ở mục 19. (Ta đã trace đệ quy bảng này ở mục 8.)

### Bài 3 (điền TODO)

Hoàn thiện hàm đếm mìn lân cận (dùng `DIRS` và kiểm tra biên đầy đủ):

In [34]:
def count_mines_todo(board, i, j):
    m, n = len(board), len(board[0])
    cnt = 0
    for dx, dy in DIRS:
        x, y = i + dx, j + dy
        # TODO: nếu (x, y) nằm trong bảng VÀ board[x][y] == "M" thì tăng cnt
        pass
    return cnt


# Khi làm xong, bỏ comment để kiểm tra:
# g = board_from(["MEE", "EEE", "EEM"])
# assert [count_mines_todo(g, 1, 1), count_mines_todo(g, 0, 1), count_mines_todo(g, 2, 0)] == [2, 1, 0]

### Bài 4 (tự viết lại)

Không nhìn lời giải, tự viết `updateBoard` bằng **BFS** hoặc **DFS**:

In [35]:
class MySolution:
    def updateBoard(self, board, click):
        pass            # TODO


# Khi làm xong:
# for b, c, w in [(board1, [3, 0], expected1), (board2, [1, 2], expected2)]:
#     assert MySolution().updateBoard(deepcopy(b), c) == w

### Bài 5 (mini challenge): thắng, thua, đang chơi

Viết `play_result(board, clicks)` áp dụng lần lượt các lần bấm lên một bảng và trả về một trong ba chuỗi: `"LOST"` nếu có `'X'` xuất hiện; `"WON"` nếu **không còn ô `'E'` nào** (mọi ô không phải mìn đã lật, mìn thì còn nguyên `'M'`); `"PLAYING"` nếu hết lần bấm mà chưa thắng/thua. Không được sửa bảng đầu vào của người gọi. Gợi ý: dùng lại lời giải chính.

In [36]:
def play_result(board, clicks):
    pass            # TODO


# Khi làm xong:
# assert play_result(board_from(["EEE", "EME", "EEE"]), [[0, 0]]) == "PLAYING"

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đếm các `'M'` trong 8 ô quanh ô cần xét.
- Mức 2: bảng là `MEE / EEE / EEM`; ô `(1,1)` ở giữa nên đủ 8 láng giềng.
- Mức 3: góc `(0,0)` là mìn nhưng `(1,1)` có hai mìn ở hai góc đối diện; ô `(2,0)` ở góc dưới trái chỉ có 3 láng giềng.

**Bài 2**

- Mức 1: mìn ở `(2,1)` chỉ ảnh hưởng 8 ô quanh nó.
- Mức 2: ô nào không kề mìn và bị lan tới thì thành `'B'`; ô kề mìn thành `'1'`.
- Mức 3: hàng 0 toàn `B`; hàng 1: `1 1 1 B`; hàng 2: `(2,0)` có láng giềng `(1,0)=1,(1,1)=1,(2,1)=M`, không kề ô `B` nào nên còn `E`.

**Bài 3**

- Mức 1: kiểm tra biên như ở mục 3.2.
- Mức 2: điều kiện `0 <= x < m and 0 <= y < n and board[x][y] == "M"`.
- Mức 3: `cnt += 1` bên trong `if` đó.

**Bài 4**

- Mức 1: trúng mìn thì dừng; còn lại lật ô bấm.
- Mức 2: chỉ ô `'B'` mới đưa vào hàng đợi; ghi giá trị ngay khi đưa vào.
- Mức 3: xem lại code ở mục 9 và tự viết lại không nhìn.

**Bài 5**

- Mức 1: `deepcopy` đầu vào trước.
- Mức 2: sau mỗi lần bấm kiểm tra có `'X'` trong bảng chưa, rồi kiểm tra còn `'E'` không.
- Mức 3: ví dụ bảng `["EEE","EME","EEE"]` với click `[0,0]` chỉ lật một ô (ra `'1'`) nên vẫn `"PLAYING"`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

`(1,1)`: hai mìn ở `(0,0)` và `(2,2)` → `2`; `(0,1)`: láng giềng gồm `(0,0)` mìn → `1`; `(2,0)`: láng giềng `(1,0),(1,1),(2,1)` → `0`; số láng giềng: góc 3, giữa 8, cạnh 5.

In [37]:
g = board_from(["MEE", "EEE", "EEM"])
out = (count_mines(g, 1, 1), count_mines(g, 0, 1), count_mines(g, 2, 0))
assert out == (2, 1, 0)
assert (len(neighbors(0, 0, 3, 3)), len(neighbors(1, 1, 3, 3)), len(neighbors(0, 1, 3, 3))) == (3, 8, 5)
print("Bài 1:", out)

Bài 1: (2, 1, 0)


### Bài 2

Kết quả tính tay:

```
B B B B
1 1 1 B
E M 1 B
```

Ô `(2, 0)` vẫn là `'E'` vì không kề ô `'B'` nào (láng giềng của nó là `(1,0)=1`, `(1,1)=1`, `(2,1)=M`).

In [38]:
board3 = board_from(["EEEE", "EEEE", "EMEE"])
want3 = board_from(["BBBB", "111B", "EM1B"])
for name, got in run_all(board3, [0, 3]).items():
    assert got == want3, name
assert is_valid(board3, want3, [0, 3])
show(want3, "Bài 2 - kết quả:")

Bài 2 - kết quả:
B B B B
1 1 1 B
E M 1 B


### Bài 3

In [39]:
def count_mines_answer(board, i, j):
    m, n = len(board), len(board[0])
    cnt = 0
    for dx, dy in DIRS:
        x, y = i + dx, j + dy
        if 0 <= x < m and 0 <= y < n and board[x][y] == "M":
            cnt += 1
    return cnt


g = board_from(["MEE", "EEE", "EEM"])
assert [count_mines_answer(g, 1, 1), count_mines_answer(g, 0, 1), count_mines_answer(g, 2, 0)] == [2, 1, 0]
rng = random.Random(1)
for _ in range(200):
    bd = random_board(rng, rng.randint(1, 6), rng.randint(1, 6), 0.3)
    for i in range(len(bd)):
        for j in range(len(bd[0])):
            assert count_mines_answer(bd, i, j) == count_mines(bd, i, j)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [40]:
class MySolutionAnswer:
    def updateBoard(self, board, click):
        m, n = len(board), len(board[0])
        i, j = click
        if board[i][j] == "M":
            board[i][j] = "X"
            return board

        def reveal(x, y):
            c = sum(1 for dx in (-1, 0, 1) for dy in (-1, 0, 1)
                    if (dx or dy) and 0 <= x + dx < m and 0 <= y + dy < n and board[x + dx][y + dy] == "M")
            board[x][y] = str(c) if c else "B"
            return board[x][y]

        if reveal(i, j) != "B":
            return board
        queue = deque([(i, j)])
        while queue:
            x, y = queue.popleft()
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    a, b = x + dx, y + dy
                    if 0 <= a < m and 0 <= b < n and board[a][b] == "E":
                        if reveal(a, b) == "B":
                            queue.append((a, b))
        return board


for b, c, w in [(board1, [3, 0], expected1), (board2, [1, 2], expected2)]:
    assert MySolutionAnswer().updateBoard(deepcopy(b), c) == w
rng = random.Random(5)
for _ in range(150):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    bd = random_partial_board(rng, m, n, 0.15, rng.randint(0, 2))
    un = [(i, j) for i in range(m) for j in range(n) if bd[i][j] in "ME"]
    if un:
        c = list(rng.choice(un))
        assert MySolutionAnswer().updateBoard(deepcopy(bd), c) == update_board_brute(deepcopy(bd), c)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [41]:
def play_result_answer(board, clicks):
    board = deepcopy(board)                         # không động vào bảng của người gọi
    for c in clicks:
        Solution().updateBoard(board, c)
        if any("X" in row for row in board):
            return "LOST"
        if not any("E" in row for row in board):
            return "WON"
    return "PLAYING"


orig = board_from(["EEE", "EME", "EEE"])
assert play_result_answer(orig, [[0, 0]]) == "PLAYING"
assert play_result_answer(orig, [[0, 0], [1, 1]]) == "LOST"
assert orig == board_from(["EEE", "EME", "EEE"])         # bảng gốc nguyên vẹn
# thắng: bảng không mìn, một lần bấm lật hết
assert play_result_answer(board_from(["EEE", "EEE"]), [[0, 0]]) == "WON"
# thắng sau nhiều nước: 8 ô quanh mìn giữa đều là số 1, bấm từng ô
all_cells = [[i, j] for i in range(3) for j in range(3) if (i, j) != (1, 1)]
assert play_result_answer(orig, all_cells) == "WON"
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template flood fill 8 hướng trên bảng (BFS):**

```python
class Solution:
    def updateBoard(self, board, click):
        m, n = len(board), len(board[0])
        i, j = click
        if board[i][j] == "M":                 # trúng mìn
            board[i][j] = "X"
            return board

        def reveal(x, y):                       # lật 1 ô: số mìn lân cận hoặc 'B'
            cnt = sum(1 for dx in (-1, 0, 1) for dy in (-1, 0, 1)
                      if (dx or dy) and 0 <= x + dx < m and 0 <= y + dy < n
                      and board[x + dx][y + dy] == "M")
            board[x][y] = str(cnt) if cnt else "B"
            return board[x][y]

        if reveal(i, j) != "B":                 # ô số: dừng
            return board
        queue = deque([(i, j)])
        while queue:
            x, y = queue.popleft()
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    a, b = x + dx, y + dy
                    if 0 <= a < m and 0 <= b < n and board[a][b] == "E":
                        if reveal(a, b) == "B": # ghi ngay = đánh dấu đã thăm
                            queue.append((a, b))
        return board
```

**Checklist trước khi submit:**

- [ ] Có xét **đủ 8 hướng** (cả đường chéo) cho cả đếm mìn và loang?
- [ ] Kiểm tra biên **cả hai phía** (`0 <= x < m`), không để chỉ số âm "quấn vòng"?
- [ ] Trúng `'M'` → `'X'` và **return ngay**?
- [ ] Ô có số: ghi `str(cnt)` (là **chuỗi**) và **không lan tiếp**?
- [ ] Ghi giá trị mới **ngay khi** đưa vào hàng đợi (không có ô nào vào hai lần)?
- [ ] Chỉ lan tới ô đang là `'E'` (không đụng `'M'`, `'B'`, chữ số)?
- [ ] Nếu dùng đệ quy: đã nâng `sys.setrecursionlimit` hoặc chuyển sang BFS?
- [ ] Khi tự test: dùng `deepcopy` và trả về chính `board`; tên class/method đúng skeleton.